# Azure OpenAI API - Hands-On Session

Welcome to this hands-on session where we'll explore:
1. **Token Counting** - Understanding and calculating tokens
2. **LLM Input/Output Optimizations** - Chain of Thought & Parameter Tuning
3. **LLM Inference Optimizations** - Quantization & KV Caching

---

## Setup and Installation

First, let's install the required packages.

In [1]:
# # Install required packages
# !pip install openai tiktoken python-dotenv transformers torch bitsandbytes accelerate -q

In [1]:
# Import libraries
import os
import tiktoken
from openai import AzureOpenAI
from dotenv import load_dotenv
import time

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Configure Azure OpenAI client
# Set your Azure OpenAI credentials here or use environment variables

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "your-endpoint-here")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "your-api-key-here")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")  # Your deployment name
AZURE_OPENAI_API_VERSION = "2025-01-01-preview"

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

print(f"Azure OpenAI client configured!")
print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Deployment: {AZURE_OPENAI_DEPLOYMENT}")

Azure OpenAI client configured!
Endpoint: https://llmappsbuild.openai.azure.com/
Deployment: gpt-4o-mini


---

# Part 1: Understanding and Counting Tokens

## What are Tokens?

Tokens are the fundamental units that LLMs use to process text. They're not exactly words or characters - they're somewhere in between.

**Key Points:**
- A token can be a word, part of a word, or punctuation
- On average, 1 token ≈ 4 characters in English
- 1 token ≈ 0.75 words
- Token count affects: **Cost**, **Latency**, and **Context Window limits**

### Why Token Counting Matters:
1. **Cost Management** - You're billed per token (input + output)
2. **Context Window** - Each model has a max token limit (e.g., GPT-4o: 128K tokens)
3. **Performance** - More tokens = longer processing time

In [3]:
# Initialize the tokenizer for GPT models
# cl100k_base is used for GPT-4, GPT-3.5-turbo, and text-embedding models

encoding = tiktoken.get_encoding("cl100k_base")

# You can also get encoding for a specific model
# encoding = tiktoken.encoding_for_model("gpt-4")

print(f"Tokenizer loaded: cl100k_base")

Tokenizer loaded: cl100k_base


In [4]:
def count_tokens(text, encoding=encoding):
    """Count the number of tokens in a text string."""
    tokens = encoding.encode(text)
    return len(tokens)

def visualize_tokens(text, encoding=encoding):
    """Visualize how text is split into tokens."""
    tokens = encoding.encode(text)
    token_strings = [encoding.decode([t]) for t in tokens]
    
    print(f"Original text: '{text}'")
    print(f"Number of tokens: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    print(f"Tokens: {token_strings}")
    print("-" * 50)
    return tokens

In [5]:
# Example 1: Simple sentence
visualize_tokens("Hello, world!")

Original text: 'Hello, world!'
Number of tokens: 4
Token IDs: [9906, 11, 1917, 0]
Tokens: ['Hello', ',', ' world', '!']
--------------------------------------------------


[9906, 11, 1917, 0]

In [6]:
# Example 2: Longer text
sample_text = "Azure OpenAI Service provides REST API access to OpenAI's powerful language models."
visualize_tokens(sample_text)

Original text: 'Azure OpenAI Service provides REST API access to OpenAI's powerful language models.'
Number of tokens: 16
Token IDs: [79207, 5377, 15836, 5475, 5825, 26487, 5446, 2680, 311, 5377, 15836, 596, 8147, 4221, 4211, 13]
Tokens: ['Azure', ' Open', 'AI', ' Service', ' provides', ' REST', ' API', ' access', ' to', ' Open', 'AI', "'s", ' powerful', ' language', ' models', '.']
--------------------------------------------------


[79207,
 5377,
 15836,
 5475,
 5825,
 26487,
 5446,
 2680,
 311,
 5377,
 15836,
 596,
 8147,
 4221,
 4211,
 13]

In [7]:
# Example 3: Notice how uncommon words get split
visualize_tokens("Pneumonoultramicroscopicsilicovolcanoconiosis")

Original text: 'Pneumonoultramicroscopicsilicovolcanoconiosis'
Number of tokens: 17
Token IDs: [47, 818, 372, 263, 11206, 99040, 2823, 2445, 454, 1233, 321, 292, 869, 337, 69377, 444, 91260]
Tokens: ['P', 'ne', 'um', 'on', 'oul', 'tram', 'icro', 'sc', 'op', 'ics', 'il', 'ic', 'ov', 'ol', 'cano', 'con', 'iosis']
--------------------------------------------------


[47,
 818,
 372,
 263,
 11206,
 99040,
 2823,
 2445,
 454,
 1233,
 321,
 292,
 869,
 337,
 69377,
 444,
 91260]

In [8]:
# Example 4: Code gets tokenized differently
code_sample = """def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)"""

visualize_tokens(code_sample)

Original text: 'def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)'
Number of tokens: 28
Token IDs: [755, 76798, 1471, 997, 262, 422, 308, 2717, 220, 16, 512, 286, 471, 308, 198, 262, 471, 76798, 1471, 12, 16, 8, 489, 76798, 1471, 12, 17, 8]
Tokens: ['def', ' fibonacci', '(n', '):\n', '   ', ' if', ' n', ' <=', ' ', '1', ':\n', '       ', ' return', ' n', '\n', '   ', ' return', ' fibonacci', '(n', '-', '1', ')', ' +', ' fibonacci', '(n', '-', '2', ')']
--------------------------------------------------


[755,
 76798,
 1471,
 997,
 262,
 422,
 308,
 2717,
 220,
 16,
 512,
 286,
 471,
 308,
 198,
 262,
 471,
 76798,
 1471,
 12,
 16,
 8,
 489,
 76798,
 1471,
 12,
 17,
 8]

In [9]:
# Counting tokens for a chat message (includes special tokens)
def count_chat_tokens(messages, model="gpt-4"):
    """Count tokens for a list of chat messages."""
    encoding = tiktoken.encoding_for_model(model)
    
    # Every message follows format: <|start|>{role/name}\n{content}<|end|>
    tokens_per_message = 3  # <|start|>, role, <|end|>
    tokens_per_name = 1
    
    total_tokens = 0
    for message in messages:
        total_tokens += tokens_per_message
        for key, value in message.items():
            total_tokens += len(encoding.encode(value))
            if key == "name":
                total_tokens += tokens_per_name
    
    total_tokens += 3  # Every reply is primed with <|start|>assistant<|message|>
    return total_tokens

# Example chat messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"}
]

print(f"Estimated tokens for chat: {count_chat_tokens(messages)}")

Estimated tokens for chat: 22


### Cost Estimation Example

In [10]:
# Cost estimation (example prices - check Azure pricing for current rates)
def estimate_cost(input_tokens, output_tokens, model="gpt-4o-mini"):
    """Estimate cost based on token count."""
    # Example pricing per 1K tokens (USD) - Update with actual Azure pricing
    pricing = {
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "gpt-5-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4": {"input": 0.03, "output": 0.06},
        "gpt-35-turbo": {"input": 0.0005, "output": 0.0015}
    }
    
    if model not in pricing:
        return "Model pricing not found"
    
    input_cost = (input_tokens / 1000) * pricing[model]["input"]
    output_cost = (output_tokens / 1000) * pricing[model]["output"]
    total_cost = input_cost + output_cost
    
    return {
        "input_cost": f"${input_cost:.6f}",
        "output_cost": f"${output_cost:.6f}",
        "total_cost": f"${total_cost:.6f}"
    }

# Example: 1000 input tokens, 500 output tokens
cost = estimate_cost(1000, 500, "gpt-5-mini")
print(f"Cost estimate for 1000 input + 500 output tokens:")
print(f"  Input cost: {cost['input_cost']}")
print(f"  Output cost: {cost['output_cost']}")
print(f"  Total cost: {cost['total_cost']}")

Cost estimate for 1000 input + 500 output tokens:
  Input cost: $0.000150
  Output cost: $0.000300
  Total cost: $0.000450


---

# Part 2: LLM Input/Output Optimizations

This section covers techniques to improve LLM outputs through:
1. **Prompt Engineering** - Chain of Thought (CoT)
2. **Parameter Tuning** - Temperature, Top-P, Penalties

---

## 2.1 Chain of Thought (CoT) Prompting

### What is Chain of Thought?

Chain of Thought prompting encourages the model to break down complex problems into intermediate reasoning steps before arriving at the final answer.

**Benefits:**
- Improved accuracy on complex reasoning tasks
- Better explainability of the model's reasoning
- Reduced errors in math, logic, and multi-step problems

**Types of CoT:**
1. **Zero-shot CoT** - Add "Let's think step by step"
2. **Few-shot CoT** - Provide examples with reasoning steps
3. **Self-Consistency** - Generate multiple reasoning paths

In [11]:
def call_azure_openai(prompt, system_message="You are a helpful assistant.", **kwargs):
    """Helper function to call Azure OpenAI API."""
    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        **kwargs
    )
    return response.choices[0].message.content

In [12]:
# Math problem to demonstrate CoT
math_problem = """A store sells apples for $2 each and oranges for $3 each. 
If John buys 4 apples and 3 oranges, and he pays with a $20 bill, how much change does he get?"""

print("=" * 60)
print("STANDARD PROMPTING (No CoT)")
print("=" * 60)

standard_response = call_azure_openai(
    prompt=math_problem + "\n\nGive me just the final answer.",
    temperature=0.1
)
print(f"Response: {standard_response}")

STANDARD PROMPTING (No CoT)
Response: John gets $11 in change.


In [13]:
print("=" * 60)
print("ZERO-SHOT CHAIN OF THOUGHT")
print("=" * 60)

cot_response = call_azure_openai(
    prompt=math_problem + "\n\nLet's think step by step.",
    temperature=0.1
)
print(f"Response:\n{cot_response}")

ZERO-SHOT CHAIN OF THOUGHT
Response:
Let's calculate the total cost of the fruits John buys step by step.

1. **Calculate the cost of the apples:**
   - John buys 4 apples.
   - The price of each apple is $2.
   - Total cost for apples = Number of apples × Price per apple
   - Total cost for apples = 4 × $2 = $8.

2. **Calculate the cost of the oranges:**
   - John buys 3 oranges.
   - The price of each orange is $3.
   - Total cost for oranges = Number of oranges × Price per orange
   - Total cost for oranges = 3 × $3 = $9.

3. **Calculate the total cost of the fruits:**
   - Total cost = Total cost for apples + Total cost for oranges
   - Total cost = $8 + $9 = $17.

4. **Calculate the change John receives:**
   - John pays with a $20 bill.
   - Change = Amount paid - Total cost
   - Change = $20 - $17 = $3.

Therefore, John gets **$3** in change.


In [14]:
# More complex reasoning problem
logic_problem = """There are 5 houses in a row, each painted a different color.
The red house is to the left of the green house.
The blue house is in the middle.
The yellow house is at one end.
The white house is to the right of the blue house.

What is the order of the houses from left to right?"""

print("=" * 60)
print("COMPLEX REASONING WITH CoT")
print("=" * 60)

cot_logic = call_azure_openai(
    prompt=logic_problem + "\n\nLet's solve this step by step, considering each constraint carefully.",
    temperature=0.1
)
print(f"Response:\n{cot_logic}")

COMPLEX REASONING WITH CoT
Response:
Let's analyze the clues step by step to determine the order of the houses.

1. **The blue house is in the middle.**
   - Since there are 5 houses, the blue house must be the 3rd house.

   ```
   1st | 2nd | 3rd (Blue) | 4th | 5th
   ```

2. **The yellow house is at one end.**
   - The yellow house can either be the 1st house or the 5th house. We will consider both possibilities.

3. **The red house is to the left of the green house.**
   - This means that the red house must come before the green house in the order.

4. **The white house is to the right of the blue house.**
   - Since the blue house is in the 3rd position, the white house must be either the 4th or the 5th house.

Now, let's consider the two cases for the yellow house:

### Case 1: Yellow house is 1st
```
1st (Yellow) | 2nd | 3rd (Blue) | 4th | 5th
```
- The white house must be in the 4th or 5th position. If the white house is in the 4th position, then the 5th position must be the gr

In [15]:
# Few-shot CoT example
few_shot_prompt = """I'll show you how to solve word problems step by step.

Example 1:
Problem: Sarah has 15 candies. She gives 1/3 to her brother. How many candies does she have left?
Solution:
Step 1: Find 1/3 of 15 candies
Step 2: 15 ÷ 3 = 5 candies given to brother
Step 3: 15 - 5 = 10 candies remaining
Answer: Sarah has 10 candies left.

Example 2:
Problem: A train travels at 60 mph. How far will it travel in 2.5 hours?
Solution:
Step 1: Use the formula: Distance = Speed × Time
Step 2: Distance = 60 mph × 2.5 hours
Step 3: Distance = 150 miles
Answer: The train will travel 150 miles.

Now solve this problem:
Problem: A rectangular garden has a length of 12 meters and a width of 8 meters. 
A path 1 meter wide runs around the outside of the garden. 
What is the area of the path?
Solution:"""

print("=" * 60)
print("FEW-SHOT CHAIN OF THOUGHT")
print("=" * 60)

few_shot_response = call_azure_openai(
    prompt=few_shot_prompt,
    temperature=0
)
print(f"Response:\n{few_shot_response}")

FEW-SHOT CHAIN OF THOUGHT
Response:
To solve the problem step by step, we will first find the area of the garden and then the area of the garden plus the path. Finally, we will subtract the area of the garden from the area of the garden plus the path to find the area of the path.

Step 1: Calculate the area of the garden.
- The area of a rectangle is given by the formula: Area = Length × Width.
- For the garden, Length = 12 meters and Width = 8 meters.
- Area of the garden = 12 meters × 8 meters = 96 square meters.

Step 2: Calculate the dimensions of the garden plus the path.
- The path is 1 meter wide and runs around the garden. This means we need to add 2 meters to both the length and the width (1 meter on each side).
- New Length = 12 meters + 2 meters = 14 meters.
- New Width = 8 meters + 2 meters = 10 meters.

Step 3: Calculate the area of the garden plus the path.
- Area of the garden plus the path = New Length × New Width = 14 meters × 10 meters = 140 square meters.

Step 4: Ca

---

## 2.2 LLM Parameters Deep Dive

Understanding these parameters is crucial for controlling LLM behavior:

| Parameter | Range | Description |
|-----------|-------|-------------|
| **Temperature** | 0-2 | Controls randomness. Lower = more deterministic |
| **Top-P** | 0-1 | Nucleus sampling. Considers tokens comprising top P probability mass |
| **Frequency Penalty** | -2 to 2 | Penalizes repeated tokens based on frequency |
| **Presence Penalty** | -2 to 2 | Penalizes tokens that have appeared at all |

---

### Temperature

Temperature controls the "creativity" or randomness of the output.

- **Temperature = 0**: Deterministic, always picks the most likely token
- **Temperature = 0.7**: Balanced creativity (good default)
- **Temperature = 1.0**: Standard sampling
- **Temperature > 1.0**: More random, creative, but may be incoherent

**Use Cases:**
- **Low (0-0.3)**: Code generation, factual Q&A, data extraction
- **Medium (0.5-0.7)**: General conversation, explanations
- **High (0.8-1.2)**: Creative writing, brainstorming

In [16]:
prompt = "Write a creative one-sentence description of a sunset."

print("=" * 60)
print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0, 0.5, 1.0, 1.5]:
    print(f"\n--- Temperature: {temp} ---")
    for i in range(3):  # Run 3 times to show variation
        response = call_azure_openai(prompt, temperature=temp, max_tokens=100)
        print(f"  {i+1}. {response}")

TEMPERATURE COMPARISON

--- Temperature: 0 ---
  1. As the sun dipped below the horizon, it painted the sky in a breathtaking tapestry of fiery oranges and soft purples, as if the day itself was whispering a gentle farewell to the world.
  2. As the sun dipped below the horizon, it painted the sky in a breathtaking tapestry of fiery oranges and soft purples, as if the day itself was whispering a gentle farewell to the night.
  3. As the sun dipped below the horizon, it painted the sky in a breathtaking tapestry of fiery oranges and soft purples, as if the day itself was whispering a gentle farewell to the world.

--- Temperature: 0.5 ---
  1. As the sun dipped below the horizon, it painted the sky in a breathtaking tapestry of fiery oranges and soft purples, casting a golden glow that whispered promises of a tranquil night to come.
  2. As the sun dipped below the horizon, it painted the sky in a breathtaking tapestry of fiery oranges and soft purples, casting a golden glow that whispe

### Top-P (Nucleus Sampling)

Top-P considers only the smallest set of tokens whose cumulative probability exceeds P.

- **Top-P = 0.1**: Only considers tokens in the top 10% probability mass (very focused)
- **Top-P = 0.9**: Considers tokens in the top 90% probability mass (more diverse)
- **Top-P = 1.0**: Considers all tokens

**Note:** Generally, modify either Temperature OR Top-P, not both. Azure recommends changing one while keeping the other at default.

In [17]:
prompt = "List 5 unusual hobbies someone might enjoy:"

print("=" * 60)
print("TOP-P COMPARISON")
print("=" * 60)

for top_p in [0.1, 0.5, 0.9]:
    print(f"\n--- Top-P: {top_p} ---")
    response = call_azure_openai(prompt, temperature=1.0, top_p=top_p, max_tokens=150)
    print(response)

TOP-P COMPARISON

--- Top-P: 0.1 ---
Sure! Here are five unusual hobbies that someone might enjoy:

1. **Soap Carving**: This involves creating intricate designs and sculptures from bars of soap. It's a unique blend of art and craftsmanship that allows for creativity and fine motor skills.

2. **Extreme Ironing**: This quirky hobby combines the mundane task of ironing clothes with extreme sports. Enthusiasts take their ironing boards to unusual locations, such as mountain tops or underwater, and document their adventures.

3. **Geocaching**: Often described as a real-world treasure hunt, geocaching involves using GPS coordinates to find hidden containers, called "geocaches," in various locations. It combines outdoor exploration with puzzle-solving.

4. **Competitive Duck Herding**:

--- Top-P: 0.5 ---
Here are five unusual hobbies that someone might enjoy:

1. **Soap Carving**: This creative hobby involves carving intricate designs and sculptures out of bars of soap. It's a relaxing ac

### Frequency Penalty

Reduces repetition by penalizing tokens based on how often they've appeared.

**Formula:** `logit_bias[token] -= frequency_penalty * token_count`

- **0**: No penalty (default)
- **Positive (0 to 2)**: Discourages repetition
- **Negative (-2 to 0)**: Encourages repetition

**Use Cases:**
- High values: Prevent repetitive text in long outputs
- Low/negative: When consistency or repeated themes are desired

In [19]:
# Using a prompt that naturally tends toward repetition
prompt = "List 10 reasons why the sky is blue. Be detailed in your explanation."

print("=" * 60)
print("FREQUENCY PENALTY COMPARISON")
print("=" * 60)

for freq_penalty in [0, 1.0, 2.0]:
    print(f"\n--- Frequency Penalty: {freq_penalty} ---")
    response = call_azure_openai(prompt, frequency_penalty=freq_penalty, max_tokens=300)
    print(response)
    # Count repetitions of key words
    words = response.lower().split()
    print(f"\n  Word counts: 'blue'={words.count('blue')}, 'sky'={words.count('sky')}, 'light'={words.count('light')}")

FREQUENCY PENALTY COMPARISON

--- Frequency Penalty: 0 ---
The blue color of the sky is primarily due to a phenomenon known as Rayleigh scattering, but several other factors also contribute to our perception of the sky’s blue color. Here are ten detailed reasons explaining why the sky appears blue:

1. **Rayleigh Scattering**: The primary reason behind the blue color of the sky is Rayleigh scattering, which occurs when sunlight enters the Earth's atmosphere. Sunlight, or white light, is composed of multiple colors. When it strikes small particles in the atmosphere, shorter wavelengths of light (blue and violet) scatter more than longer wavelengths (red and yellow). Although violet light scatters even more than blue, our eyes are more sensitive to blue light, and some violet is absorbed by the ozone layer, making the sky appear predominantly blue.

2. **Sun’s Position**: The position of the Sun in the sky influences the color we perceive. When the Sun is high overhead, the light travels

### Presence Penalty

Similar to frequency penalty, but applies a one-time penalty for tokens that have appeared at all.

**Formula:** `logit_bias[token] -= presence_penalty * (1 if token_appeared else 0)`

- **0**: No penalty (default)
- **Positive (0 to 2)**: Encourages talking about new topics
- **Negative (-2 to 0)**: Encourages staying on topic

**Difference from Frequency Penalty:**
- Frequency: Penalizes based on HOW MANY times a token appeared
- Presence: Penalizes based on WHETHER a token appeared (binary)

In [20]:
prompt = "Tell me about cats. Then continue telling me more."

print("=" * 60)
print("PRESENCE PENALTY COMPARISON")
print("=" * 60)

for presence_penalty in [0, 1.0, 2.0]:
    print(f"\n--- Presence Penalty: {presence_penalty} ---")
    response = call_azure_openai(
        prompt, 
        temperature=0.7, 
        presence_penalty=presence_penalty, 
        max_tokens=200
    )
    print(response)

PRESENCE PENALTY COMPARISON

--- Presence Penalty: 0 ---
Cats, scientifically known as Felis catus, are small, domesticated mammals that belong to the family Felidae. They are one of the most popular pets around the world, cherished for their companionship, playful behavior, and independent nature. 

### Physical Characteristics
Cats typically weigh between 5 to 20 pounds, depending on the breed. They have a flexible body, sharp retractable claws, and keen senses—especially sight and hearing. Cats have a wide range of coat colors and patterns, from solid black or white to tabby stripes and calico patches.

### Behavior
Cats are known for their playful and curious behavior. They often engage in hunting-like play, stalking and pouncing on toys or even their human companions. They have a unique way of communicating that includes vocalizations (meows, purrs, hisses), body language, and facial expressions. 

### Social Structure
While cats are often seen as solitary animals, they can form s

In [21]:
# Combined parameter demonstration
print("=" * 60)
print("PARAMETER COMBINATIONS")
print("=" * 60)

prompt = "Write a short poem about the ocean."

configs = [
    {"name": "Precise & Consistent", "temperature": 0.2, "top_p": 0.9, "frequency_penalty": 0, "presence_penalty": 0},
    {"name": "Balanced & Varied", "temperature": 0.7, "top_p": 0.9, "frequency_penalty": 0.5, "presence_penalty": 0.5},
    {"name": "Creative & Diverse", "temperature": 1.0, "top_p": 0.95, "frequency_penalty": 1.0, "presence_penalty": 1.0},
]

for config in configs:
    print(f"\n--- {config['name']} ---")
    print(f"    temp={config['temperature']}, top_p={config['top_p']}, freq={config['frequency_penalty']}, pres={config['presence_penalty']}")
    response = call_azure_openai(
        prompt,
        temperature=config["temperature"],
        top_p=config["top_p"],
        frequency_penalty=config["frequency_penalty"],
        presence_penalty=config["presence_penalty"],
        max_tokens=150
    )
    print(response)

PARAMETER COMBINATIONS

--- Precise & Consistent ---
    temp=0.2, top_p=0.9, freq=0, pres=0
Beneath the sky, a canvas wide,  
The ocean whispers, swells with pride.  
Its azure depths, a world unseen,  
Where secrets dance in shades of green.  

Waves embrace the golden shore,  
A rhythmic song, forevermore.  
With salty breeze and sunlit gleam,  
It cradles dreams, a sailor's dream.  

In twilight's glow, the waters gleam,  
Reflecting stars, a silver stream.  
The ocean calls, both fierce and free,  
A timeless heart, a mystery.  

--- Balanced & Varied ---
    temp=0.7, top_p=0.9, freq=0.5, pres=0.5
Beneath the sky, a canvas wide,  
The ocean whispers, deep and tide.  
With waves that dance in sunlit grace,  
A liquid mirror, a vast embrace.  

Its secrets held in depths profound,  
Where ancient songs of silence sound.  
Blue horizon meets the day,  
In foamy frolics, children play.  

The salty breeze sings tales untold,  
Of ships and dreams and treasures bold.  
Oh ocean vast, 

### Parameter Selection Guidelines

| Task Type | Temperature | Top-P | Freq Penalty | Pres Penalty |
|-----------|-------------|-------|--------------|---------------|
| Code Generation | 0-0.2 | 0.95 | 0 | 0 |
| Data Extraction | 0 | 1.0 | 0 | 0 |
| Chatbot | 0.5-0.7 | 0.9 | 0.5 | 0.5 |
| Creative Writing | 0.8-1.0 | 0.95 | 0.5-1.0 | 0.5-1.0 |
| Brainstorming | 1.0-1.2 | 0.95 | 1.0 | 1.0 |

---

# Part 3: LLM Inference Optimizations

This section covers techniques to make LLM inference faster and more efficient:

1. **Model Quantization** - Reducing model precision to decrease memory and increase speed
2. **KV Caching** - Caching key-value pairs to avoid redundant computations

**Note:** For these demos, we'll use HuggingFace Transformers with a small local model.

---

## 3.1 Model Quantization

### What is Quantization?

Quantization reduces the precision of model weights from higher precision (FP32/FP16) to lower precision (INT8/INT4).

**Benefits:**
- Reduced memory footprint (up to 4x smaller)
- Faster inference (less computation)
- Lower hardware requirements

**Trade-offs:**
- Slight quality degradation (usually minimal)
- Some quantization methods require calibration

**Common Quantization Types:**
- **FP32** (Full Precision): 32 bits per parameter
- **FP16** (Half Precision): 16 bits per parameter
- **INT8**: 8 bits per parameter (~4x smaller than FP32)
- **INT4**: 4 bits per parameter (~8x smaller than FP32)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc

# We'll use a small model for demonstration
MODEL_NAME = "microsoft/phi-2"  # 2.7B parameter model, good for demos
# Alternative smaller models: "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "facebook/opt-350m"

print(f"Using model: {MODEL_NAME}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using model: microsoft/phi-2
PyTorch version: 2.9.1
CUDA available: False


In [2]:
def get_model_size(model):
    """Calculate model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

def measure_inference_time(model, tokenizer, prompt, num_runs=3):
    """Measure average inference time."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    
    # Measure
    times = []
    for _ in range(num_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.time() - start)
    
    return sum(times) / len(times), tokenizer.decode(outputs[0], skip_special_tokens=True)

In [3]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

test_prompt = "Explain what machine learning is in simple terms:"

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [1]:
# Load FP16 model (baseline)
print("=" * 60)
print("Loading FP16 Model (Half Precision)")
print("=" * 60)

# Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

fp16_size = get_model_size(model_fp16)
print(f"FP16 Model Size: {fp16_size:.2f} MB")

# Measure inference
fp16_time, fp16_output = measure_inference_time(model_fp16, tokenizer, test_prompt)
print(f"FP16 Inference Time: {fp16_time:.3f}s")
print(f"Output: {fp16_output[:200]}...")

# Clean up
del model_fp16
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading FP16 Model (Half Precision)


NameError: name 'torch' is not defined

In [7]:
# Load 8-bit quantized model
print("=" * 60)
print("Loading INT8 Quantized Model")
print("=" * 60)

quantization_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config_8bit,
    device_map="auto",
    trust_remote_code=True
)

int8_size = get_model_size(model_int8)
print(f"INT8 Model Size: {int8_size:.2f} MB")
print(f"Size Reduction: {((fp16_size - int8_size) / fp16_size * 100):.1f}%")

# Measure inference
int8_time, int8_output = measure_inference_time(model_int8, tokenizer, test_prompt)
print(f"INT8 Inference Time: {int8_time:.3f}s")
print(f"Speedup: {fp16_time / int8_time:.2f}x")
print(f"Output: {int8_output[:200]}...")

# Clean up
del model_int8
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading INT8 Quantized Model


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


INT8 Model Size: 2901.83 MB
Size Reduction: 45.3%


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


INT8 Inference Time: 23.738s
Speedup: 1.25x
Output: Explain what machine learning is in simple terms:

Machine learning is a type of artificial intelligence that allows computers to learn from data and make predictions or decisions without being explic...


In [8]:
# Load 4-bit quantized model (most aggressive quantization)
print("=" * 60)
print("Loading INT4 Quantized Model (NF4)")
print("=" * 60)

quantization_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",  # NormalFloat4 - better for normally distributed weights
    bnb_4bit_use_double_quant=True  # Nested quantization for more savings
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config_4bit,
    device_map="auto",
    trust_remote_code=True
)

int4_size = get_model_size(model_int4)
print(f"INT4 Model Size: {int4_size:.2f} MB")
print(f"Size Reduction vs FP16: {((fp16_size - int4_size) / fp16_size * 100):.1f}%")

# Measure inference
int4_time, int4_output = measure_inference_time(model_int4, tokenizer, test_prompt)
print(f"INT4 Inference Time: {int4_time:.3f}s")
print(f"Speedup vs FP16: {fp16_time / int4_time:.2f}x")
print(f"Output: {int4_output[:200]}...")

# Keep this model for KV caching demo
model = model_int4

Loading INT4 Quantized Model (NF4)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


INT4 Model Size: 1701.83 MB
Size Reduction vs FP16: 67.9%


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


INT4 Inference Time: 15.173s
Speedup vs FP16: 1.96x
Output: Explain what machine learning is in simple terms:

Machine learning is a type of artificial intelligence that allows computers to learn and make decisions without being explicitly programmed. It invol...


In [ ]:
# Summary comparison
print("=" * 60)
print("QUANTIZATION SUMMARY")
print("=" * 60)
print(f"\n{'Precision':<12} {'Size (MB)':<12} {'Time (s)':<12} {'Relative Size':<15} {'Speedup'}")
print("-" * 60)
print(f"{'FP16':<12} {fp16_size:<12.2f} {fp16_time:<12.3f} {'100%':<15} {'1.00x'}")
print(f"{'INT8':<12} {int8_size:<12.2f} {int8_time:<12.3f} {f'{int8_size/fp16_size*100:.1f}%':<15} {f'{fp16_time/int8_time:.2f}x'}")
print(f"{'INT4':<12} {int4_size:<12.2f} {int4_time:<12.3f} {f'{int4_size/fp16_size*100:.1f}%':<15} {f'{fp16_time/int4_time:.2f}x'}")

---

## 3.2 KV Caching

### What is KV Caching?

In transformer models, attention mechanism computes Key (K) and Value (V) matrices for each token. During autoregressive generation, we can **cache** previously computed KV pairs instead of recomputing them.

**How it works:**
1. First token: Compute K, V for input prompt → Cache them
2. Second token: Compute K, V only for new token → Append to cache
3. Repeat: Each new token only needs O(1) new KV computation

**Benefits:**
- Significantly faster generation for long sequences
- Reduces redundant computation from O(n²) to O(n)

**Trade-offs:**
- Increased memory usage (storing the cache)
- Cache size grows with sequence length

In [ ]:
def generate_without_kv_cache(model, tokenizer, prompt, max_new_tokens=50):
    """Generate text without KV caching (recompute everything each step)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    
    start_time = time.time()
    
    for _ in range(max_new_tokens):
        with torch.no_grad():
            # Full forward pass without using cache
            outputs = model(input_ids, use_cache=False)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            
            # Check for EOS
            if next_token.item() == tokenizer.eos_token_id:
                break
                
            input_ids = torch.cat([input_ids, next_token], dim=-1)
    
    elapsed = time.time() - start_time
    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    tokens_generated = input_ids.shape[1] - inputs["input_ids"].shape[1]
    
    return generated_text, elapsed, tokens_generated

In [ ]:
def generate_with_kv_cache(model, tokenizer, prompt, max_new_tokens=50):
    """Generate text with KV caching (incremental generation)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    past_key_values = None
    
    start_time = time.time()
    
    for i in range(max_new_tokens):
        with torch.no_grad():
            if past_key_values is None:
                # First pass: process entire prompt
                outputs = model(input_ids, use_cache=True)
            else:
                # Subsequent passes: only process new token
                outputs = model(input_ids[:, -1:], past_key_values=past_key_values, use_cache=True)
            
            past_key_values = outputs.past_key_values
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            
            # Check for EOS
            if next_token.item() == tokenizer.eos_token_id:
                break
                
            input_ids = torch.cat([input_ids, next_token], dim=-1)
    
    elapsed = time.time() - start_time
    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    tokens_generated = input_ids.shape[1] - inputs["input_ids"].shape[1]
    
    return generated_text, elapsed, tokens_generated

In [ ]:
# Compare KV caching performance
print("=" * 60)
print("KV CACHING COMPARISON")
print("=" * 60)

test_prompt = "Write a detailed explanation of how neural networks learn:"
max_tokens = 30  # Reduce for faster demo

print(f"\nPrompt: {test_prompt}")
print(f"Max new tokens: {max_tokens}")

# Without KV cache
print("\n--- Without KV Cache ---")
text_no_cache, time_no_cache, tokens_no_cache = generate_without_kv_cache(
    model, tokenizer, test_prompt, max_tokens
)
print(f"Time: {time_no_cache:.3f}s")
print(f"Tokens generated: {tokens_no_cache}")
print(f"Tokens/second: {tokens_no_cache/time_no_cache:.2f}")

# With KV cache
print("\n--- With KV Cache ---")
text_with_cache, time_with_cache, tokens_with_cache = generate_with_kv_cache(
    model, tokenizer, test_prompt, max_tokens
)
print(f"Time: {time_with_cache:.3f}s")
print(f"Tokens generated: {tokens_with_cache}")
print(f"Tokens/second: {tokens_with_cache/time_with_cache:.2f}")

print(f"\n--- Speedup: {time_no_cache/time_with_cache:.2f}x ---")

In [ ]:
# Visualize KV cache memory usage
print("=" * 60)
print("KV CACHE MEMORY ANALYSIS")
print("=" * 60)

def estimate_kv_cache_size(model, seq_length, batch_size=1):
    """Estimate KV cache size in MB."""
    # Get model config
    config = model.config
    num_layers = getattr(config, 'num_hidden_layers', getattr(config, 'n_layer', 32))
    num_heads = getattr(config, 'num_attention_heads', getattr(config, 'n_head', 32))
    head_dim = getattr(config, 'hidden_size', 2048) // num_heads
    
    # KV cache size: 2 (K+V) * num_layers * batch_size * num_heads * seq_length * head_dim * bytes_per_element
    bytes_per_element = 2  # FP16
    cache_size = 2 * num_layers * batch_size * num_heads * seq_length * head_dim * bytes_per_element
    return cache_size / (1024 ** 2)  # Convert to MB

print(f"\nEstimated KV Cache Size for Different Sequence Lengths:")
print(f"{'Seq Length':<15} {'Cache Size (MB)':<20}")
print("-" * 35)
for seq_len in [128, 256, 512, 1024, 2048, 4096]:
    cache_size = estimate_kv_cache_size(model, seq_len)
    print(f"{seq_len:<15} {cache_size:<20.2f}")

In [ ]:
# Using HuggingFace's built-in generate with cache
print("=" * 60)
print("Using HuggingFace's Optimized Generate")
print("=" * 60)

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

# HuggingFace automatically uses KV caching in generate()
start = time.time()
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    use_cache=True  # This is True by default
)
hf_time = time.time() - start

print(f"HuggingFace generate() time: {hf_time:.3f}s")
print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)[:300]}...")

---

## Summary: Optimization Techniques

### Input/Output Optimizations
| Technique | What it Does | When to Use |
|-----------|--------------|-------------|
| Chain of Thought | Breaks down reasoning | Complex reasoning, math, logic |
| Temperature | Controls randomness | Low for precision, high for creativity |
| Top-P | Nucleus sampling | Alternative to temperature |
| Frequency Penalty | Reduces repetition | Long-form content |
| Presence Penalty | Encourages new topics | Diverse outputs |

### Inference Optimizations
| Technique | Benefit | Trade-off |
|-----------|---------|----------|
| INT8 Quantization | ~2x smaller, faster | Minimal quality loss |
| INT4 Quantization | ~4x smaller, faster | Slight quality loss |
| KV Caching | Much faster generation | Uses more memory |

### Best Practices
1. **Start with defaults**, then tune parameters based on output quality
2. **Use quantization** when deploying to resource-constrained environments
3. **Always enable KV caching** for autoregressive generation
4. **Combine techniques** - e.g., INT4 + KV caching for maximum efficiency

In [ ]:
# Cleanup
print("Cleaning up resources...")
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Done!")

---

## Additional Resources

- [Azure OpenAI Documentation](https://learn.microsoft.com/en-us/azure/ai-services/openai/)
- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)
- [HuggingFace Transformers](https://huggingface.co/docs/transformers/)
- [BitsAndBytes Quantization](https://github.com/TimDettmers/bitsandbytes)

---

**Happy Learning!**